In [1]:
pip install torch sentence-transformers pinecone langgraph sendgrid langchain-groq pytesseract fitz

In [ ]:
# nonLLM_function.py

import os
from dotenv import load_dotenv
from sendgrid import SendGridAPIClient
from sendgrid.helpers.mail import Mail, From
from pinecone import Pinecone
from sentence_transformers import SentenceTransformer
import time
import logging

load_dotenv()

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')


try:
    pc = Pinecone(api_key=os.getenv('PINECONE_API_KEY'))
    index = pc.Index("customer-support-rag")
    emb_model = SentenceTransformer("all-MiniLM-L6-v2")
except Exception as e:
    logging.error(f"Failed to initialize Pinecone or SentenceTransformer: {e}")
    index = None
    emb_model = None

def log_ticket(user_query, llm_response, confidence, param4, category):
    """
    Log ticket data to Pinecone for future RAG retrieval.
    """
    if not index or not emb_model:
        logging.warning("Pinecone or embedding model not initialized. Skipping ticket logging.")
        return

    logging.info("Logging ticket to Pinecone...")

    ticket_content = f"Query: {user_query}\nResponse: {llm_response}"
    try:
        embedding = emb_model.encode(ticket_content).tolist()
        ticket_id = f"ticket_{hash(user_query)}_{int(time.time())}"

        metadata = {
            "text": ticket_content,
            "category": category,
            "timestamp": int(time.time()),
            "status": "closed" if category != "escalated" else "escalated"
        }

        index.upsert(vectors=[{
            "id": ticket_id,
            "values": embedding,
            "metadata": metadata
        }])
        logging.info(f"Stored ticket {ticket_id} in Pinecone with category: {category}")
    except Exception as e:
        logging.error(f"Failed to upsert ticket to Pinecone: {e}")

def send_email_and_log(to_email, query, response, confidence, category):

    logging.info("Sending email via SendGrid and logging to Pinecone...")

    subject = f"Support Response: {category.capitalize()} Inquiry"
    html_content = (
        f"<p>Dear Customer,</p>"
        f"<p>Thank you for reaching out with your query: <strong>{query}</strong></p>"
        f"<p>{response.replace('\n', '<br>')}</p>"
        f"<p>If you need further assistance, please reply or contact us at "
        f"<a href='mailto:support@my_email.com'>support@my_email.com</a>.</p>"
        f"<p>Best regards,<br>Your Support Team</p>"
    )

    message = Mail(
        from_email=From('neerajmaurya015@gmail.com'),
        to_emails=to_email,
        subject=subject,
        html_content=html_content
    )
    message.reply_to = 'support@my_email.com'

    try:
        sg = SendGridAPIClient(os.getenv('SENDGRID_API_KEY'))
        sg_response = sg.send(message)
        logging.info(f"Email sent to {to_email} with status: {sg_response.status_code}")
    except Exception as e:
        logging.error(f"Email sending failed: {e}")
        if hasattr(e, 'body'):
            logging.error(f"Error details: {e.body}")

    log_ticket(query, response, confidence, True, category=category)
    logging.info(f"Email sent to {to_email} and ticket logged.")

def escalate_ticket_to_human(user_query, llm_response, to_email, escalated):

    agent_email = 'neerajmaurya1017@gmail.com'
    category = "escalated"

    logging.info("Escalating ticket to human...")

    # Notify customer
    customer_subject = "Your Support Ticket Has Been Escalated"
    customer_body = (
        f"<p>Dear Customer,</p>"
        f"<p>Your query: <strong>{user_query}</strong></p>"
        f"<p>Initial response: {llm_response.replace('\n', '<br>')}</p>"
        f"<p>We've escalated this to a specialist. Expect a response within 24 hours.</p>"
        f"<p>Best regards,<br>Support Team</p>"
    )
    message_to_customer = Mail(
        from_email=From('neerajmaurya015@gmail.com', 'Support Team'),
        to_emails=to_email,
        subject=customer_subject,
        html_content=customer_body
    )
    message_to_customer.reply_to = 'support@my_email.com'


    # Notify agent
    agent_subject = "New Escalated Ticket"
    agent_body = (
        f"<p>Escalated Ticket:</p>"
        f"<p>Customer Email: {to_email}</p>"
        f"<p>Query: <strong>{user_query}</strong></p>"
        f"<p>LLM Response: {llm_response.replace('\n', '<br>')}</p>"
        f"<p>Category: {category}</p>"
        f"<p>Please resolve and reply to customer, then log resolution.</p>"
    )

    message_to_agent = Mail(
        from_email=From('neerajmaurya015@gmail.com', 'Support Team'),
        to_emails=agent_email,
        subject=agent_subject,
        html_content=agent_body
    )
    message_to_agent.reply_to = to_email

    try:
        sg = SendGridAPIClient(os.getenv('SENDGRID_API_KEY'))
        sg.send(message_to_customer)
        sg.send(message_to_agent)
        logging.info(f"Escalation emails sent to {to_email} and {agent_email}.")
    except Exception as e:
        logging.error(f"Escalation email failed: {e}")
        if hasattr(e, 'body'):
            logging.error(f"Error details: {e.body}")

    log_ticket(user_query, llm_response, 0.0, False, category="escalated")

In [ ]:
# llm_function.py

import logging

from pinecone import Pinecone
from sentence_transformers import SentenceTransformer
import os
from dotenv import load_dotenv
load_dotenv()

try:
    pc = Pinecone(api_key=os.getenv('PINECONE_API_KEY'))
    index = pc.Index("customer-support-rag")
    emb_model = SentenceTransformer("all-MiniLM-L6-v2")
except Exception as e:
    print(f"Failed to initialize Pinecone: {e}")
    index = None
    emb_model = None

def search_documents(query, category="billing", top_k=3):
    if not index or not emb_model:
        print("Pinecone or embedding model not initialized. Returning empty results.")
        return []


    query_embedding = emb_model.encode(query).tolist()

    try:
        results = index.query(
            vector=query_embedding,
            top_k=top_k,
            include_metadata=True,
            filter={"category": {"$eq": category}}
        )
        formatted_results = [
            (match["metadata"]["text"], match["metadata"], match["score"])
            for match in results["matches"]
        ]
        print(f"Searched Pinecone for {top_k} documents in category: {category}")
        print(f"Results: {formatted_results}")
        return formatted_results
    except Exception as e:
        print(f"Pinecone query failed: {e}")
        return []


# --------- LLM functions ---------


def query_classifier(query: str) -> str:
    import os

    if not os.getenv('GROQ_API_KEY'):
       raise RuntimeError("GROQ_API_KEY not set in env")

    from langchain_groq import ChatGroq
    llm = ChatGroq(
    model="meta-llama/llama-4-maverick-17b-128e-instruct",
    api_key=os.environ.get('GROQ_API_KEY'),
    temperature=0.7
    )
    prompt = f"Classify this query into [billing, technical, account, security]: {query} . Always give one word answer "

    cat = llm.invoke(prompt).content
    print(cat)
    return  cat.lower()



def generate_response(query, retrieved_docs):
    import os

    if not os.getenv('GROQ_API_KEY'):
       raise RuntimeError("GROQ_API_KEY not set in env")

    from langchain_groq import ChatGroq

    llm = ChatGroq(
    model="meta-llama/llama-4-maverick-17b-128e-instruct",
    api_key=os.environ.get('GROQ_API_KEY'),
    temperature=0.7
    )

    context = "\n".join([d[0] for d in retrieved_docs])
    prompt = f"""
    You are a helpful SaaS support assistant.
    User query: {query}

    Relevant knowledge base context:
    {context}

    Please provide a clear, friendly, and accurate response from the relevant knowledge base , please don't hallucinate.
    """

    response = llm.invoke(prompt).content

    return response



def critic_agent(query, response, retrieved_docs):
    sim_threshold=0.75
    import os
    from sentence_transformers import SentenceTransformer

    from scipy.spatial import distance
    import re
    emb_model = SentenceTransformer("all-MiniLM-L6-v2")  # output is 384-dim embeddings
    resp_emb = emb_model.encode(response)
    sims = [1 - distance.cosine(resp_emb, emb_model.encode(doc[0])) for doc in retrieved_docs]

    if not sims:
        return False, {"reason": "no_docs", "sims": sims}

    max_sim = max(sims)

    #Semantic grounding check
    if max_sim < sim_threshold:
        return False, {"reason": "low_similarity", "sims": sims}

    #Hallucination check (checking atleast any keyword matched or not)
    for kw in ["meet", "zoom", "login", "password", "invoice", "refund", "subscription", "account", "error", "bug", "feature"]:
        if kw in response.lower() and not any(kw in d[0].lower() for d in retrieved_docs):
            return False, {"reason": f"hallucinated_{kw}", "sims": sims}

    #Tone check
    if re.search(r"(idiot|stupid|useless)", response.lower()):
        return False, {"reason": "toxic_tone", "sims": sims}

    #Confidence check LLM
    llm_confidence = critic_llm_confidence(query,response)
    if llm_confidence is not None and llm_confidence < 0.6:
        return False, {"reason": "low_llm_confidence", "sims": sims}

    #Passed all checks
    return True, {"reason": "valid", "sims": sims}


def critic_llm_confidence(query, response):
    import os

    if not os.getenv('GROQ_API_KEY'):
       raise RuntimeError("GROQ_API_KEY not set in env")

    from langchain_groq import ChatGroq

    llm = ChatGroq(
    model="meta-llama/llama-3-8b-8192",
    api_key=os.environ.get('GROQ_API_KEY'),
    temperature=0.7
    )

    prompt = f"""
    On a scale of 0 to 1, how confident are you that the following response accurately addresses the user's query based on provided context?
    User query: {query}
    Response: {response}
    Provide only a decimal number between 0 and 1.
    """

    confidence_resp = llm.invoke(prompt).content
    try:
        confidence_score = float(confidence_resp.strip())
        return confidence_score
    except ValueError:
        return None

In [ ]:
# customer_support_pipeline.py

import os
from dotenv import load_dotenv
import logging

load_dotenv()

# Set up logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

from nonLLM_function import (
    log_ticket,
    send_email_and_log,
    escalate_ticket_to_human
)
from llm_function import (
    generate_response,
    search_documents,
    critic_agent,
    query_classifier
)

from langgraph.graph import StateGraph, START, END
from typing import TypedDict

class TicketState(TypedDict):
    user_query: str
    category: str
    search_docs: list
    llm_response: str
    send_mail: bool
    details: dict
    conversation_history: list
    escalated: bool

# -------- Nodes -------
def ticket_ingest(state: TicketState):
    query = state["user_query"]
    logging.info(f"Ticket ingested with query: {query}")
    return {"user_query": query}

def classifier(state: TicketState):
    q = state["user_query"]
    cat = query_classifier(q)
    logging.info(f"Query classified as: {cat}")
    return {"category": cat}

def retriever(state: TicketState):
    category = state["category"]
    docs = search_documents(state["user_query"], category=category, top_k=3)
    logging.info(f"Retrieved documents for category {category}: {len(docs)} docs")
    return {"search_docs": docs}

def responder(state: TicketState):
    context = "\n".join([f"User: {turn['user']}\nBot: {turn.get('bot', '')}" for turn in state.get("conversation_history", [])])
    full_query = f"{context}\nUser: {state['user_query']}" if context else state["user_query"]
    logging.info(f"Generating response with full query: {full_query[:100]}...")
    resp = generate_response(full_query, state["search_docs"])
    print("Generated response:", resp)
    return {"llm_response": resp}

def critic(state: TicketState):
    send_reply, details = critic_agent(
        state["user_query"], state["llm_response"], state["search_docs"]
    )
    logging.info(f"Critic decision: send_reply={send_reply}, details={details}")
    return {"send_mail": send_reply, "details": details}

def escalate_human(state: TicketState):
    escalate_ticket_to_human(
        state["user_query"],
        state["llm_response"],
        to_email="neerajmaurya0015@gmail.com",
        escalated=True
    )
    logging.info("Ticket escalated to human")
    return {"escalated": True}

def send_reply(state: TicketState):
    send_email_and_log(
        to_email="neerajmaurya0015@gmail.com",
        query=state["user_query"],
        response=state["llm_response"],
        confidence=state["details"].get("confidence", 1.0),
        category=state["category"]
    )
    history = state.get("conversation_history", [])
    history.append({"user": state["user_query"], "bot": state["llm_response"]})
    logging.info(f"Reply sent, history updated to length {len(history)}")
    return {"conversation_history": history}

def decision_node(state: TicketState):
    return "send_reply" if state["send_mail"] else "escalate"

# ----- Build Graph -------
graph = StateGraph(TicketState)

graph.add_node("ticket_ingest", ticket_ingest)
graph.add_node("classifier", classifier)
graph.add_node("retriever", retriever)
graph.add_node("responder", responder)
graph.add_node("critic", critic)
graph.add_node("escalate", escalate_human)
graph.add_node("send_reply", send_reply)

graph.set_entry_point("ticket_ingest")

graph.add_edge("ticket_ingest", "classifier")
graph.add_edge("classifier", "retriever")
graph.add_edge("retriever", "responder")
graph.add_edge("responder", "critic")
graph.add_conditional_edges(
    "critic",
    decision_node,
    {"send_reply": "send_reply", "escalate": "escalate"}
)
graph.add_edge("send_reply", END)
graph.add_edge("escalate", END)

workflow = graph.compile()

# Multi-Turn Loop
max_turns = 5
conversation_history = []
user_email = "neerajmaurya015@gmail.com"
state = {"conversation_history": conversation_history, "escalated": False}

while True:
    query = input("Enter your support query (or 'exit' to close): ")
    if query.lower() == 'exit':
        print("Support session ended.")
        break

    state["user_query"] = query
    try:
        response = workflow.invoke(state)
        # Update state with the response from the workflow
        state.update(response)
    except Exception as e:
        logging.error(f"Pipeline failed: {e}")
        break

    if state.get("escalated", False):
        print("Ticket escalated to human. Awaiting resolution...")
        human_response = input("Human Agent: Enter resolution response (or 'close' to end): ")
        if human_response != 'close':
            send_email_and_log(
                to_email=user_email,
                query=query,
                response=human_response,
                confidence=1.0,
                category=state.get("category", "unknown")
            )
        print("Ticket closed after human resolution.")
        break
    else:
        # History is already updated in state from send_reply
        print("conversation history:", state.get("conversation_history", []))
        follow_up = input("Bot response sent. Enter follow-up query (or 'done' to close): ")
        if follow_up.lower() == 'done':
            print("Ticket closed automatically.")
            break
        state["user_query"] = follow_up

In [ ]:
# onetime_injectData_to_pinecone.py

import fitz
import pytesseract
from PIL import Image
import io
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.schema import Document


pdf_paths = {
    "billing": "Billing-Zoom-Guide.pdf",
    "account": "Account-Zoom-User-Account-Settings.pdf",
    "technical": "technical-issues-Zoom-Trouble-Shooting-Guide.pdf",
    "security": "other-issues-security-onboarding-guide.pdf"
}

all_docs = []
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)

for category, path in pdf_paths.items():
    # Load text with PyPDFLoader
    loader = PyPDFLoader(path)
    docs = loader.load()

    # Extract images and text with PyMuPDF
    pdf_doc = fitz.open(path)
    for page_num in range(len(pdf_doc)):
        page_text = docs[page_num].page_content
        print(page_num)

        page = pdf_doc[page_num]
        image_list = page.get_images(full=True)
        for img_index, img in enumerate(image_list):
            xref = img[0]
            base_image = pdf_doc.extract_image(xref)
            image_bytes = base_image["image"]
            image = Image.open(io.BytesIO(image_bytes))

            # Option 1: Use OCR to extract text from images
            image_text = pytesseract.image_to_string(image)

            # Append image-derived text to page content
            if image_text.strip():
                page_text += f"\nImage {img_index+1} content: {image_text}"

        # Update the document's content with augmented text
        docs[page_num].page_content = page_text
        docs[page_num].metadata["category"] = category

    # Chunk the augmented documents
    chunks = splitter.split_documents(docs)
    all_docs.extend(chunks)

    pdf_doc.close()

# print(all_docs[0])



import os
from dotenv import load_dotenv
load_dotenv()
#import pinecone
from langchain.vectorstores import Pinecone
from pinecone import Pinecone, ServerlessSpec

pc = Pinecone(api_key=os.getenv('PINECONE_API_KEY'))
index_name = "customer-support-rag"


if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )
index = pc.Index(index_name)


from sentence_transformers import SentenceTransformer

# Load free local embedding model
emb_model = SentenceTransformer("all-MiniLM-L6-v2")

# Assume all_docs is your list of documents (e.g., from LangChain loader)
# all_docs = [...]  # Your pre-chunked docs with page_content and metadata

# Generate embeddings and prepare vectors (one-time)
vectors = []
for i, doc in enumerate(all_docs):
    chunk = doc.page_content
    embedding = emb_model.encode(chunk).tolist()  # Free local embedding

    # Copy metadata and add 'text' for storage
    metadata = doc.metadata.copy()  # Preserves original metadata incl. 'category'
    metadata['text'] = chunk  # Store chunk text for easy retrieval in RAG

    # Unique ID (use source + page if available, or simple index)
    doc_id = f"doc_{i}_{metadata.get('source', 'unknown').replace('/', '_').replace('.', '_')}_{metadata.get('page', i)}"
    print(doc_id)
    vectors.append({
        "id": doc_id,
        "values": embedding,
        "metadata": metadata
    })

# Batch size to avoid 2MB limit
batch_size = 100
for j in range(0, len(vectors), batch_size):
    batch = vectors[j:j + batch_size]
    index.upsert(vectors=batch)
    print(f"Upserted batch {j // batch_size + 1} of {len(vectors) // batch_size + 1}")

print("All documents embedded and stored in Pinecone with metadata.")